<a href="https://colab.research.google.com/github/GustavoKozan/IC-UFPR-Agents/blob/main/Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# CÉLULA 1: INSTALAÇÃO E CONFIGURAÇÃO DO AMBIENTE DE BENCHMARK (CORRIGIDA)
# ==============================================================================
import os
import sys
import urllib.request
import subprocess

print("Instalando dependências do benchmark...")

# 1. Instalação das dependências analíticas padrão
!pip install -q requests tqdm pandas openpyxl tabulate radon

# 2. Descarrega e corrige a sintaxe legada do setup.py do HumanEval
!rm -rf human-eval
!git clone -q https://github.com/openai/human-eval.git
!sed -i 's/human_eval.evaluate_functional_correctness/human_eval.evaluate_functional_correctness:main/g' human-eval/setup.py

# 3. Instalação robusta do pacote usando subprocess do Python
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "./human-eval"], check=True)
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "./human-eval", "--no-build-isolation"], check=True)

# 4. Localização exata do pacote carregado e download garantido do dataset no caminho correto
try:
    import human_eval
    from human_eval.data import read_problems

    package_dir = os.path.dirname(human_eval.__file__)
    # O pacote busca em: human_eval/../data/HumanEval.jsonl.gz
    target_dir = os.path.abspath(os.path.join(package_dir, "..", "data"))
    os.makedirs(target_dir, exist_ok=True)

    target_file = os.path.join(target_dir, "HumanEval.jsonl.gz")
    url = "https://github.com/openai/human-eval/raw/master/data/HumanEval.jsonl.gz"

    print(f"Baixando HumanEval.jsonl.gz para:\n{target_file}")
    urllib.request.urlretrieve(url, target_file)

    # Teste prático de leitura das 164 tarefas
    problemas_teste = read_problems()
    print(f"\n✓ Framework HumanEval instalado e validado ({len(problemas_teste)} tarefas prontas)!")
    print("✓ Ambiente 100% operacional para executar as Células 2 e 3.")
except Exception as err:
    raise RuntimeError(f"Falha ao validar instalação do human-eval: {err}")

Instalando dependências do benchmark...
Baixando HumanEval.jsonl.gz para:
/usr/local/lib/python3.13/dist-packages/data/HumanEval.jsonl.gz

✓ Framework HumanEval instalado e validado (164 tarefas prontas)!
✓ Ambiente 100% operacional para executar as Células 2 e 3.


In [ ]:
# ==============================================================================
# CÉLULA 2: MOTOR UNIVERSAL DE INFERÊNCIA COM RACIOCÍNIO NATIVO DESATIVADO
# ==============================================================================
import json
import os
from pathlib import Path
import re
import textwrap
import time
from google.colab import userdata
from human_eval.data import read_problems
import requests
from tqdm import tqdm

# ------------------------------------------------------------------------------
# 1. ESCOLHA DO MODELO (Altere apenas esta linha)
# ------------------------------------------------------------------------------
MODEL_NAME = "z-ai/glm-5.3-flash"
PROVIDER = "auto"

# ------------------------------------------------------------------------------
# 2. CONFIGURAÇÃO DOS PROVEDORES E LIMITES
# ------------------------------------------------------------------------------
CONFIGS = {
    "google": {
        "url": "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions",
        "keys": ["GEMINI_API_KEY", "GOOGLE_API_KEY"],
        "delay": 4.5,
        "timeout": 90.0,
    },
    "nvidia": {
        "url": "https://integrate.api.nvidia.com/v1/chat/completions",
        "keys": ["NVIDIA_API_KEY"],
        "delay": 5.5,
        "timeout": 240.0,  # 240s para acomodar eventual cold-start de protótipos
    },
    "groq": {
        "url": "https://api.groq.com/openai/v1/chat/completions",
        "keys": ["GROQ_API_KEY"],
        "delay": 1.5,
        "timeout": 60.0,
    },
    "openrouter": {
        "url": "https://openrouter.ai/api/v1/chat/completions",
        "keys": ["OPENROUTER_API_KEY"],
        "delay": 4.5,
        "timeout": 90.0,
    },
}

def obter_secret_seguro(nome_chave: str):
    try:
        val = userdata.get(nome_chave)
        if val:
            return str(val).strip().replace('"', "").replace("'", "")
    except Exception:
        pass
    return None

def resolver_provedor(modelo: str, selecao_manual: str) -> str:
    if selecao_manual != "auto":
        return selecao_manual.lower()
    m = modelo.lower()
    if m.startswith("gemini"):
        return "google"
    elif "groq" in m or "openai/gpt-oss" in m:
        return "groq"
    elif any(m.startswith(p) for p in [
        "moonshotai/", "nvidia/", "meta/", "deepseek-ai/",
        "mistralai/", "poolside/", "qwen/", "microsoft/", "ibm/"
    ]):
        return "nvidia"
    elif obter_secret_seguro("NVIDIA_API_KEY"):
        return "nvidia"
    elif obter_secret_seguro("GEMINI_API_KEY") or obter_secret_seguro("GOOGLE_API_KEY"):
        return "google"
    return "openrouter"

provedor_ativo = resolver_provedor(MODEL_NAME, PROVIDER)
cfg = CONFIGS.get(provedor_ativo, CONFIGS["google"])

api_key = None
for k in cfg["keys"]:
    api_key = obter_secret_seguro(k)
    if api_key:
        break

if not api_key:
    raise ValueError(f"Chave de autenticação não localizada para [{provedor_ativo.upper()}]!")

INVOKE_URL = cfg["url"]
DELAY_CALLS = cfg["delay"]
TIMEOUT_CALL = cfg["timeout"]

headers = {
    "Authorization": f"Bearer {api_key}",
    "Accept": "application/json",
    "Content-Type": "application/json",
}

MODEL_SLUG = re.sub(r"[^a-zA-Z0-9_-]", "_", MODEL_NAME)
output_samples = Path(f"samples_{MODEL_SLUG}.jsonl")
output_metrics = Path(f"metrics_cache_{MODEL_SLUG}.json")

# ------------------------------------------------------------------------------
# 3. PROMPT ESTREITO (PROIBIÇÃO DE CHAIN-OF-THOUGHT/RACIOCÍNIO)
# ------------------------------------------------------------------------------
SYSTEM_PROMPT = (
    "Você é um motor de geração de código Python determinístico.\n"
    "REGRAS OBRIGATÓRIAS:\n"
    "1. NÃO gere raciocínio, reflexão interna, explicações, pensamentos ou cadeia de raciocínio.\n"
    "2. Comece IMEDIATAMENTE com o bloco ```python e encerre com ```.\n"
    "3. NÃO altere o nome da função nem a sua assinatura."
)

def sanitize_completion(raw_text: str, original_prompt: str) -> str:
    if not raw_text or not raw_text.strip():
        return ""

    raw_text = raw_text.replace("\xa0", " ")

    cleaned = re.sub(r"<think>.*?</think>", "", raw_text, flags=re.DOTALL)
    cleaned = re.sub(r"```thinking\s*\n?.*?```", "", cleaned, flags=re.DOTALL).strip()
    if not cleaned:
        cleaned = raw_text

    matches = list(re.finditer(r"```(?:python)?\s*\n?(.*?)(?:```|$)", cleaned, flags=re.DOTALL))
    clean = matches[-1].group(1).strip() if matches else cleaned.strip()

    func_match = re.search(r"def\s+([a-zA-Z0-9_]+)\s*\(", original_prompt)
    if func_match:
        func_name = func_match.group(1)
        lines = clean.splitlines()
        def_line_idx = -1
        for idx, line in enumerate(lines):
            if re.match(rf"^\s*def\s+{func_name}\b", line):
                def_line_idx = idx
                break

        if def_line_idx != -1:
            lines = lines[def_line_idx + 1:]
            while lines and not lines[0].strip():
                lines = lines[1:]
            if lines and (lines[0].strip().startswith('"""') or lines[0].strip().startswith("'''")):
                quote = lines[0].strip()[:3]
                lines = lines[1:]
                while lines:
                    if quote in lines[0]:
                        lines = lines[1:]
                        break
                    lines = lines[1:]
            clean = "\n".join(lines)

    dedented = textwrap.dedent(clean).strip("\n")
    return textwrap.indent(dedented, "    ") if dedented else ""

# ------------------------------------------------------------------------------
# 4. CONSTRUÇÃO DO PAYLOAD COM PARÂMETROS DE SUPRESSÃO DE RACIOCÍNIO
# ------------------------------------------------------------------------------
def construir_payload(prompt_text: str):
    corpo_user = f"Complete a seguinte função em Python:\n\n{prompt_text}"

    # Modelos multimodais ou v4.1 exigem o formato de lista de blocos
    if "flash" in MODEL_NAME.lower() or "vision" in MODEL_NAME.lower():
        content_val = [{"type": "text", "text": corpo_user}]
    else:
        content_val = corpo_user

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": content_val}
        ],
        "temperature": 0.1,
        "max_tokens": 8192,
        "stream": False,
    }

    if provedor_ativo == "google":
        payload["thinking_config"] = {"thinking_budget": 0}
    elif provedor_ativo in ["groq", "openrouter"]:
        payload["reasoning_effort"] = "low"

    return payload

# ------------------------------------------------------------------------------
# 5. TESTE DE CONECTIVIDADE
# ------------------------------------------------------------------------------
print(f"Validando conexão com [{MODEL_NAME}] em [{provedor_ativo.upper()}] (aguardando eventual cold-start)...")

payload_teste = construir_payload("print('ok')")
try:
    resp = requests.post(INVOKE_URL, headers=headers, json=payload_teste, timeout=TIMEOUT_CALL)

    if resp.status_code != 200:
        raise RuntimeError(f"HTTP {resp.status_code}: {resp.text}")
    print("✓ Conexão validada com sucesso! Raciocínio nativo desativado.\n")
except Exception as e:
    raise RuntimeError(f"Falha de conexão com {provedor_ativo.upper()}: {e}")

# ------------------------------------------------------------------------------
# 6. INFERÊNCIA DETERMINÍSTICA
# ------------------------------------------------------------------------------
def call_api_safe(prompt_text: str, max_tentativas=6):
    for tentativa in range(max_tentativas):
        try:
            payload = construir_payload(prompt_text)
            start = time.perf_counter()
            response = requests.post(INVOKE_URL, headers=headers, json=payload, timeout=TIMEOUT_CALL)
            latency = time.perf_counter() - start

            if response.status_code == 429:
                retry_h = response.headers.get("Retry-After")
                try:
                    espera = float(retry_h) if retry_h else max(15.0, 15.0 * (1.3 ** tentativa))
                except ValueError:
                    espera = max(15.0, 15.0 * (1.3 ** tentativa))
                print(f"\n[Taxa Excedida (429)] Aguardando {espera:.1f}s...")
                time.sleep(espera)
                continue

            if response.status_code != 200:
                time.sleep(4.0 * (tentativa + 1))
                continue

            data = response.json()
            choices = data.get("choices", [])
            if not choices:
                time.sleep(4.0 * (tentativa + 1))
                continue

            msg_obj = choices[0].get("message", {}) or {}
            msg = msg_obj.get("content", "") or ""

            usage = data.get("usage", {}) or {}
            return (
                msg,
                latency,
                usage.get("prompt_tokens", 0),
                usage.get("completion_tokens", 0),
                usage.get("total_tokens", 0),
            )
        except Exception as e:
            if tentativa == max_tentativas - 1:
                raise e
            time.sleep(4.0 * (tentativa + 1))

    return "", 0.0, 0, 0, 0

# ------------------------------------------------------------------------------
# 7. EXECUÇÃO INCREMENTAL DAS 164 TAREFAS
# ------------------------------------------------------------------------------
amostras_salvas = {}
metrics_log = {}

if output_samples.exists():
    with open(output_samples, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    d = json.loads(line)
                    amostras_salvas[d["task_id"]] = d["completion"]
                except Exception:
                    pass

if output_metrics.exists():
    try:
        with open(output_metrics, "r", encoding="utf-8") as f:
            metrics_log = json.load(f)
    except Exception:
        metrics_log = {}

try:
    all_problems = read_problems()
except Exception:
    all_problems = read_problems("HumanEval.jsonl.gz")

total_tarefas = len(all_problems)
tarefas_concluidas = {t_id for t_id, comp in amostras_salvas.items() if comp and str(comp).strip()}
tarefas_pendentes = {k: v for k, v in all_problems.items() if k not in tarefas_concluidas}

print(f"Executando Benchmark para '{MODEL_NAME}' [{provedor_ativo.upper()}]:")
print(f"• Escopo total: {total_tarefas} tarefas")
print(f"• Tarefas em cache: {len(tarefas_concluidas)}")
print(f"• Tarefas pendentes: {len(tarefas_pendentes)}\n")

tmp_samples = Path(f"{output_samples}.tmp")
tmp_metrics = Path(f"{output_metrics}.tmp")

for task_id, problem in tqdm(
    tarefas_pendentes.items(),
    total=len(tarefas_pendentes),
    desc=f"Processando {MODEL_SLUG[:18]}",
):
    try:
        raw, lat, p, c, t = call_api_safe(problem["prompt"])
        comp = sanitize_completion(raw, problem["prompt"])

        metrics_log[task_id] = {
            "latency_sec": lat,
            "prompt_tokens": p,
            "completion_tokens": c,
            "total_tokens": t,
            "tentativas": 1,
        }
        amostras_salvas[task_id] = comp
    except Exception as exc:
        print(f"\n[Falha de execução na tarefa {task_id}]: {exc}")
        amostras_salvas[task_id] = ""
        metrics_log[task_id] = {
            "latency_sec": 0.0,
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "total_tokens": 0,
            "tentativas": 0,
            "erro": str(exc),
        }

    with open(tmp_samples, "w", encoding="utf-8") as f:
        for tid, c_text in amostras_salvas.items():
            f.write(json.dumps({"task_id": tid, "completion": c_text}) + "\n")
    os.replace(tmp_samples, output_samples)

    with open(tmp_metrics, "w", encoding="utf-8") as f:
        json.dump(metrics_log, f)
    os.replace(tmp_metrics, output_metrics)

    time.sleep(DELAY_CALLS)

print(f"\n✓ Execução finalizada! Dados salvos em '{output_samples.name}'.")

Validando conexão com [z-ai/glm-5.3-flash] em [NVIDIA] (aguardando eventual cold-start)...
✓ Conexão validada com sucesso! Raciocínio nativo desativado.

Executando Benchmark para 'z-ai/glm-5.3-flash' [NVIDIA]:
• Escopo total: 164 tarefas
• Tarefas em cache: 0
• Tarefas pendentes: 164



Processando z-ai_glm-5_3-flash: 100%|██████████| 164/164 [1:32:01<00:00, 33.67s/it]


✓ Execução finalizada! Dados salvos em 'samples_z-ai_glm-5_3-flash.jsonl'.


In [ ]:
# ==============================================================================
# CÉLULA 3: AVALIAÇÃO DE PASS@1, MÉTRICAS RADON E EXPORTAÇÃO (ESTABILIZADA)
# ==============================================================================
import json
from pathlib import Path
import re
import sys
from human_eval.data import read_problems
from human_eval.execution import check_correctness
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
import pandas as pd
from radon.complexity import cc_visit
from radon.metrics import h_visit, mi_visit
from radon.raw import analyze
from tabulate import tabulate
from tqdm import tqdm

# 1. Identificação dinâmica do modelo avaliado na Célula 2
try:
    TARGET_MODEL = MODEL_NAME
except NameError:
    TARGET_MODEL = "gemini-3.5-flash-lite"

TARGET_SLUG = re.sub(r"[^a-zA-Z0-9_-]", "_", TARGET_MODEL)
samples_path = Path(f"samples_{TARGET_SLUG}.jsonl")
metrics_path = Path(f"metrics_cache_{TARGET_SLUG}.json")

# Fallback automático caso o arquivo tenha sido salvo com sufixo '_reteste'
if not samples_path.exists():
    alt_samples = Path(f"samples_{TARGET_SLUG}_reteste.jsonl")
    alt_metrics = Path(f"metrics_cache_{TARGET_SLUG}_reteste.json")
    if alt_samples.exists():
        samples_path = alt_samples
        metrics_path = alt_metrics
    else:
        raise FileNotFoundError(
            f"Arquivo '{samples_path}' não encontrado! Execute a Célula 2 primeiro."
        )

# 2. Carregamento seguro das métricas de telemetria e amostras
perf_metrics = {}
if metrics_path.exists():
    try:
        with open(metrics_path, "r", encoding="utf-8") as f:
            perf_metrics = json.load(f)
    except Exception:
        perf_metrics = {}

samples_dict = {}
with open(samples_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            try:
                item = json.loads(line)
                samples_dict[item["task_id"]] = item.get("completion", "")
            except Exception:
                pass

try:
    all_problems = read_problems()
except Exception:
    all_problems = read_problems("HumanEval.jsonl.gz")

print(
    f"Avaliando {len(samples_dict)} tarefas registradas para '{TARGET_MODEL}' "
    f"(Arquivo: {samples_path.name})..."
)

# 3. Execução dos testes unitários oficiais (Pass@1)
eval_results = []
for t_id, problem_data in tqdm(
    all_problems.items(), total=len(all_problems), desc="Avaliando Pass@1"
):
    comp = samples_dict.get(t_id, "")
    if comp and str(comp).strip():
        res = check_correctness(problem_data, comp, timeout=3.0)
    else:
        res = {
            "task_id": t_id,
            "passed": False,
            "result": "failed: empty completion",
            "completion_id": None,
        }
    eval_results.append(res)

df_eval = pd.DataFrame(eval_results)
df_eval["completion"] = df_eval["task_id"].map(lambda x: samples_dict.get(x, ""))

# 4. Cálculo das métricas de Engenharia de Software (Radon)
radon_records = []
for _, row in df_eval.iterrows():
    t_id = row["task_id"]
    comp = row["completion"]
    full_code = all_problems[t_id]["prompt"] + (comp if comp else "")

    cc_score = 1
    mi_score = 0.0
    h_volume = 0.0
    h_difficulty = 0.0
    sloc = 0

    if comp and str(comp).strip():
        try:
            cc_blocks = cc_visit(full_code)
            cc_score = max([b.complexity for b in cc_blocks]) if cc_blocks else 1
            mi_score = mi_visit(full_code, multi=True)
            h_metrics = h_visit(full_code)
            h_volume = h_metrics.total.volume
            h_difficulty = h_metrics.total.difficulty
            raw_metrics = analyze(full_code)
            sloc = raw_metrics.sloc
        except Exception:
            pass

    perf = perf_metrics.get(t_id, {})
    raw_res = (
        str(row.get("result", "")).replace("\n", " ").replace("\r", " ").strip()
    )

    radon_records.append({
        "ID_Tarefa": t_id,
        "Status": "Aprovado" if bool(row["passed"]) else "Falhou",
        "Resultado_Execucao": raw_res[:50],
        "Latencia_Segundos": round(float(perf.get("latency_sec", 0.0)), 2),
        "Tokens_Prompt": int(perf.get("prompt_tokens", 0)),
        "Tokens_Resposta": int(perf.get("completion_tokens", 0)),
        "Total_Tokens": int(perf.get("total_tokens", 0)),
        "Complexidade_Ciclomatica": int(cc_score),
        "Indice_Manutenibilidade": round(float(mi_score), 2),
        "Volume_Halstead": round(float(h_volume), 2),
        "Dificuldade_Halstead": round(float(h_difficulty), 2),
        "Linhas_Codigo_SLOC": int(sloc),
    })

# 5. Ordenação e Consolidação Estatística
df_final = pd.DataFrame(radon_records)
df_final["_ordem"] = df_final["ID_Tarefa"].apply(
    lambda x: int(x.split("/")[-1]) if "/" in str(x) else 0
)
df_final = (
    df_final.sort_values(by="_ordem")
    .drop(columns=["_ordem"])
    .reset_index(drop=True)
)

total_tasks = len(df_final)
total_passed = int((df_final["Status"] == "Aprovado").sum())
pass_rate = (total_passed / total_tasks * 100) if total_tasks > 0 else 0.0
error_rate = 100.0 - pass_rate

resumo_geral = [
    {
        "Métrica": "Total de Tarefas Avaliadas",
        "Valor": total_tasks,
        "Unidade": "tarefas",
    },
    {
        "Métrica": "Soluções Aprovadas (Pass@1)",
        "Valor": f"{pass_rate:.2f}%",
        "Unidade": f"{total_passed}/{total_tasks}",
    },
    {
        "Métrica": "Taxa de Falha",
        "Valor": f"{error_rate:.2f}%",
        "Unidade": f"{total_tasks - total_passed}/{total_tasks}",
    },
    {
        "Métrica": "Latência Média por Requisição",
        "Valor": round(df_final["Latencia_Segundos"].mean(), 2),
        "Unidade": "segundos",
    },
    {
        "Métrica": "Total de Tokens Processados",
        "Valor": int(df_final["Total_Tokens"].sum()),
        "Unidade": "tokens",
    },
    {
        "Métrica": "Tokens Médios de Resposta",
        "Valor": round(df_final["Tokens_Resposta"].mean(), 1),
        "Unidade": "tokens/tarefa",
    },
    {
        "Métrica": "Complexidade Ciclomática Média",
        "Valor": round(df_final["Complexidade_Ciclomatica"].mean(), 2),
        "Unidade": "escala CC",
    },
    {
        "Métrica": "Índice de Manutenibilidade Médio",
        "Valor": round(df_final["Indice_Manutenibilidade"].mean(), 2),
        "Unidade": "escala 0-100",
    },
    {
        "Métrica": "Volume Halstead Médio",
        "Valor": round(df_final["Volume_Halstead"].mean(), 2),
        "Unidade": "escala Halstead",
    },
    {
        "Métrica": "Dificuldade Halstead Média",
        "Valor": round(df_final["Dificuldade_Halstead"].mean(), 2),
        "Unidade": "escala Halstead",
    },
    {
        "Métrica": "SLOC Médio (Linhas de Código)",
        "Valor": round(df_final["Linhas_Codigo_SLOC"].mean(), 1),
        "Unidade": "linhas",
    },
]
df_resumo = pd.DataFrame(resumo_geral)

# 6. Exportação dos Relatórios (.csv e .xlsx)
csv_filename = f"relatorio_{TARGET_SLUG}_analitico.csv"
excel_filename = f"relatorio_{TARGET_SLUG}_completo.xlsx"
df_final.to_csv(csv_filename, sep=";", index=False, encoding="utf-8-sig")

wb = Workbook()
ws_res = wb.active
ws_res.title = "Resumo Executivo"
for r in dataframe_to_rows(df_resumo, index=False, header=True):
    ws_res.append(r)

ws_tar = wb.create_sheet(title="Métricas por Tarefa")
for r in dataframe_to_rows(df_final, index=False, header=True):
    ws_tar.append(r)
wb.save(excel_filename)

# 7. Impressão Formatada no Console
print("\n" + "=" * 80)
print(f"           RESUMO EXECUTIVO: {TARGET_MODEL}")
print("=" * 80)
print(
    tabulate(df_resumo, headers="keys", tablefmt="fancy_grid", showindex=False)
)
print(f"\n✓ Relatórios salvos: '{csv_filename}' e '{excel_filename}'.")

Avaliando 164 tarefas registradas para 'z-ai/glm-5.3-flash' (Arquivo: samples_z-ai_glm-5_3-flash.jsonl)...


Avaliando Pass@1: 100%|██████████| 164/164 [00:04<00:00, 34.41it/s]



           RESUMO EXECUTIVO: z-ai/glm-5.3-flash
╒══════════════════════════════════╤═════════╤═════════════════╕
│ Métrica                          │ Valor   │ Unidade         │
╞══════════════════════════════════╪═════════╪═════════════════╡
│ Total de Tarefas Avaliadas       │ 164     │ tarefas         │
├──────────────────────────────────┼─────────┼─────────────────┤
│ Soluções Aprovadas (Pass@1)      │ 95.12%  │ 156/164         │
├──────────────────────────────────┼─────────┼─────────────────┤
│ Taxa de Falha                    │ 4.88%   │ 8/164           │
├──────────────────────────────────┼─────────┼─────────────────┤
│ Latência Média por Requisição    │ 28.17   │ segundos        │
├──────────────────────────────────┼─────────┼─────────────────┤
│ Total de Tokens Processados      │ 133665  │ tokens          │
├──────────────────────────────────┼─────────┼─────────────────┤
│ Tokens Médios de Resposta        │ 575.5   │ tokens/tarefa   │
├──────────────────────────────────┼─────

In [ ]:
# ==============================================================================
# BLOCO DE VERIFICAÇÃO RÁPIDA: TESTE DE DISPONIBILIDADE DO MODELO (ONLINE CHECK)
# ==============================================================================
import time
from google.colab import userdata
import requests

# 1. Usa o MODEL_NAME da Célula 2 se já estiver carregado, ou define manualmente aqui
try:
    modelo_alvo = MODEL_NAME
except NameError:
    modelo_alvo = "deepseek-ai/deepseek-v4.1-flash"  # Altere aqui se rodar antes da Célula 2

# 2. Leitura blindada de Secrets (evita SecretNotFoundError no encadeamento)
def ler_secret_seguro(nome_chave: str):
    try:
        val = userdata.get(nome_chave)
        if val:
            return str(val).strip().replace('"', "").replace("'", "")
    except Exception:
        pass
    return None

# 3. Roteamento automático para testar no endpoint correto do modelo
m_lower = modelo_alvo.lower()
if m_lower.startswith("gemini"):
    prov_teste = "GOOGLE AI STUDIO"
    url_teste = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
    chaves_busca = ["GEMINI_API_KEY", "GOOGLE_API_KEY"]
elif "groq" in m_lower or "openai/gpt-oss" in m_lower:
    prov_teste = "GROQ CLOUD"
    url_teste = "https://api.groq.com/openai/v1/chat/completions"
    chaves_busca = ["GROQ_API_KEY"]
elif "/" in m_lower:
    prov_teste = "NVIDIA NIM"
    url_teste = "https://integrate.api.nvidia.com/v1/chat/completions"
    chaves_busca = ["NVIDIA_API_KEY"]
else:
    prov_teste = "OPENROUTER"
    url_teste = "https://openrouter.ai/api/v1/chat/completions"
    chaves_busca = ["OPENROUTER_API_KEY", "GEMINI_API_KEY", "GOOGLE_API_KEY"]

api_key_test = None
for k in chaves_busca:
    api_key_test = ler_secret_seguro(k)
    if api_key_test:
        break

if not api_key_test:
    raise ValueError(
        f"Nenhuma chave encontrada nos Secrets para {prov_teste}! "
        f"Verifique se uma destas chaves está ativa: {chaves_busca}"
    )

headers_test = {
    "Authorization": f"Bearer {api_key_test}",
    "Accept": "application/json",
    "Content-Type": "application/json",
}

payload_test = {
    "model": modelo_alvo,
    "messages": [{"role": "user", "content": "print('ok')"}],
    "max_tokens": 15,
    "temperature": 0.1,
    "stream": False,
}

# 4. Disparo de verificação com medição de latência
print(f"Testando disponibilidade de '{modelo_alvo}' em [{prov_teste}]...")
try:
    t_inicio = time.perf_counter()
    r = requests.post(url_teste, headers=headers_test, json=payload_test, timeout=240.0)
    latencia_ms = (time.perf_counter() - t_inicio) * 1000

    print(f"Status HTTP: {r.status_code} ({latencia_ms:.0f} ms)")
    if r.status_code == 200:
        print(f"✓ Modelo '{modelo_alvo}' online e aceito com sucesso!")
    else:
        print(f"✗ Resposta de erro do servidor:\n{r.text}")
except requests.exceptions.RequestException as err:
    print(f"✗ Falha de rede ou timeout ao contactar o servidor:\n{err}")

Testando disponibilidade de 'deepseek-ai/deepseek-v4.1-flash' em [NVIDIA NIM]...
✗ Falha de rede ou timeout ao contactar o servidor:
HTTPSConnectionPool(host='integrate.api.nvidia.com', port=443): Read timed out. (read timeout=120.0)


In [ ]:
from google.colab import userdata
import requests

api_key = str(userdata.get("NVIDIA_API_KEY")).strip().replace('"', "")
headers = {"Authorization": f"Bearer {api_key}"}

# Consulta direta ao registo de rotas ativas na NVIDIA NIM
r = requests.get("https://integrate.api.nvidia.com/v1/models", headers=headers, timeout=15.0)

if r.status_code == 200:
    todos_modelos = [m["id"] for m in r.json().get("data", [])]
    modelos_glm = [m for m in todos_modelos if "glm" in m.lower()]

    if modelos_glm:
        print("✓ Modelos GLM encontrados no catálogo ativo:")
        for m in modelos_glm:
            print(f"  • {m}")
    else:
        print("✗ O modelo 'z-ai/glm-5-3-flash' não consta no catálogo ativo da NVIDIA NIM.")
        print(f"Total de modelos disponíveis na sua conta: {len(todos_modelos)}")
else:
    print(f"Erro ao consultar catálogo: HTTP {r.status_code} - {r.text}")

✓ Modelos GLM encontrados no catálogo ativo:
  • z-ai/glm-5.3
  • z-ai/glm-5.3-flash
